In [1]:
from datasets import load_dataset

ds = load_dataset("millawell/wikipedia_field_of_science")['train']

/opt/anaconda3/envs/api-data-generate/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# longest = max(len(x['text']) for x in ds)
len(ds)

303900

In [112]:
ds[:2]

{'category': ['Natural_sciences', 'Natural_sciences'],
 'text': ["  Robert Cecil Hayes\n  \n  Robert Cecil Hayes (19 January 1900–3 September 1977) was a New Zealand astronomer, seismologist and organist. He was born in Wellington, New Zealand on 19 January 1900.\n  \n  Hayes did not have any formal qualifications, however he began work at the Dominion Observatory, Wellington (then named the Hector Observatory) in 1920. The observatory was responsible for maintaining the New Zealand Government Time Service. Hayes was trained in the astronomical observations necessary to keep the clocks accurate. Some of his initial work also involved working on the seismograph that the observatory inherited from seismologist George Hogben. Hayes became acting-director of the observatory after Charles Edward Adams retired from the position in 1936. He was acting-director for 12 years before the position was formalised. \n  \n  One of the contributions Hayes had while at the Observatory was the applicati

In [98]:
# check duplicated ids
import pandas as pd
df_hf = ds.to_pandas()
print(f"HF total: {len(df_hf)}")
print(f"HF unique id: {df_hf['id'].nunique()}")

HF total: 303900
HF unique id: 226386


## Process wikipedia into ATOMIC lists, store in lancedb vector database

In [ ]:
import re
import lancedb
import pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

# --- Configuration ---
DB_PATH = "./lancedb_data"
TABLE_NAME = "wiki_science_final"
MODEL_NAME = 'all-MiniLM-L6-v2'
BATCH_SIZE = 128 

def fast_get_first_five_sentences(text, topic_to_remove=None):
    """
    1. Removes potential topic prefix from the content.
    2. Normalizes whitespace and newlines.
    3. Extracts top 5 clean sentences.
    """
    if not text or not isinstance(text, str):
        return ""

    parts = text.split('\n', 1)
    if len(parts) > 1:
        content_body = parts[1]
    else:
        content_body = parts[0]
    content_body = content_body.replace('\n', ' ').replace('\r', ' ')
    
    sentences = re.split(r'(?<=[.!?])\s+', content_body.strip())
    
    clean_sentences = [s.strip() for s in sentences if len(s.strip()) > 2][:5]
    
    return " ".join(clean_sentences)

def rebuild_extreme_pipeline(ds):
    """
    Aggressive Optimization: Clean -> Consolidate -> Vectorize -> Drop Originals
    """
    # 1. Load and Deduplicate
    print("📦 Loading and deduplicating...")
    df = ds.to_pandas()
    df = df.drop_duplicates(subset=['id'], keep='first').copy()
    
    # 2. Transform into "Clean Context"
    print("📝 Transforming into Clean Context (Topic + 5 Sentences)...")
    # We combine them directly and discard the original long content column
    df['search_text'] = df['text'].apply(fast_get_first_five_sentences)
    
    # DROP the original content column to save memory and disk space
    df = df.drop(columns=['text'])
    print("🗑️ Original content dropped. Memory optimized.")

    # 3. Integrity Check
    assert df['search_text'].iloc[0].count(" ") > 5, "Clean context seems too short or empty!"

    # 4. Vectorization
    db = lancedb.connect(DB_PATH)
    model = SentenceTransformer(MODEL_NAME)
    
    print(f"🚀 Vectorizing {len(df)} records...")
    all_vectors = []
    
    for i in tqdm(range(0, len(df), BATCH_SIZE)):
        batch_texts = df['search_text'].iloc[i : i + BATCH_SIZE].tolist()
        embeddings = model.encode(batch_texts, show_progress_bar=False)
        all_vectors.extend(embeddings.tolist())
    
    df['vector'] = all_vectors

    # 5. Save Final Table
    print(f"💾 Saving to {TABLE_NAME}...")
    # This table now ONLY contains your clean context and its vector
    table = db.create_table(TABLE_NAME, data=df, mode="overwrite")
    
    # 6. FTS Index
    print("🏗️ Building FTS index...")
    table.create_fts_index("search_text", replace=True)

    print(f"✅ DONE! Total unique clean records: {len(df)}")
    return df

df = rebuild_extreme_pipeline(ds)

In [ ]:
# reformat and store in a new table, without modifying the old one (for safety)
import lancedb
import pyarrow as pa

# 1. connect and read table
db = lancedb.connect("./lancedb_data")
old_table = db.open_table("wiki_science_final")
pa_table = old_table.to_arrow()

# 2. transform the search_text into a new format: "topic: content"
def safe_transform(text_array):
    topics = []
    contents = []
    for text in text_array.to_pylist():
        import re
        parts = re.split(r'\s{2,}', text, maxsplit=1)
        
        topic = parts[0].strip()
        body = parts[1].strip() if len(parts) > 1 else ""
        
        topics.append(topic)
        contents.append(f"{topic}: {body}")
    return topics, contents

new_topics, new_search_texts = safe_transform(pa_table["search_text"])

# 3. Construct a new Arrow Table with the new columns
new_pa_table = pa_table.append_column("topic", pa.array(new_topics))
new_pa_table = new_pa_table.set_column(
    new_pa_table.schema.get_field_index("search_text"),
    "search_text",
    pa.array(new_search_texts)
)

# 4.Save to a new table (for safety, we don't overwrite the old one)
NEW_TABLE_NAME = "wiki_science_v2_fixed"
db.create_table(NEW_TABLE_NAME, data=new_pa_table, mode="create") # mode="create"

print(f"✅ {NEW_TABLE_NAME} generated!")

In [ ]:
# skip duplicate records by checking existing IDs in LanceDB before processing
import lancedb

def get_remaining_data_from_hf(ds, table_path="./lancedb_data", table_name="wiki_science_v2_fixed"):
    # 1. Connect to DB and get existing IDs
    db = lancedb.connect(table_path)
    table = db.open_table(table_name)
    # Convert to set for O(1) lookup
    existing_ids = set(table.to_pandas()['id'].tolist())

    print(f"🔎 Found {len(existing_ids)} records already in LanceDB.")

    # 2. Filter the Hugging Face Dataset
    # We use .filter() which is highly optimized for HF Datasets
    # This will keep only rows whose ID is NOT in our processed set
    ds_remaining = ds.filter(lambda row: str(row['id']) not in existing_ids)
    
    print(f"🚀 {len(ds_remaining)} records left to process.")
    return ds_remaining


In [94]:
ds_remaining = get_remaining_data_from_hf(ds)

🔎 Found 226392 records already in LanceDB.
🚀 0 records left to process.


In [ ]:
rebuild_extreme_pipeline(ds_remaining)

🚀 Starting Streaming ETL to LanceDB...


  0%|          | 0/1 [00:00<?, ?it/s]python(14136) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(14137) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(14138) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(14139) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
100%|██████████| 1/1 [00:04<00:00,  4.15s/it]

✅ Success! Data persisted in wiki_science


In [143]:
import lancedb
import pandas as pd

db = lancedb.connect("./lancedb_data")
table = db.open_table("wiki_science_v2_fixed")

# FTS index
table.create_fts_index("search_text", replace=True)

## Upload to AWS S3

In [141]:
import lancedb

db = lancedb.connect("./lancedb_data")
table = db.open_table("wiki_science_v2_fixed")

# comparct files
table.compact_files()

/var/folders/rb/h96xs37s4mb01tk65wryvqk00000gn/T/ipykernel_90512/4237487420.py:7: DeprecatedWarning: compact_files is deprecated as of 0.21.0. Use `Table.optimize` instead.
  table.compact_files()


CompactionMetrics(fragments_removed=0, fragments_added=0, files_removed=0, files_added=0)

In [ ]:
! cd ./lancedb_data && ls -lh
! aws s3 sync ./wiki_science_v2_fixed.lance s3://child-rag-bucket/backups/wiki_science_v2_fixed.lance

## Test vector search

In [137]:
# 1. Define question
query_vector = model.encode("Why is the sky blue?")

# 2. Search in LanceDB using cosine similarity
table = db.open_table('wiki_science_v2_fixed')
results = table.search(query_vector) \
    .metric("cosine") \
    .limit(5) \
    .to_pandas()

# 3. Results
print("--- Results ---")
print(results[['topic','search_text', '_distance']])

--- Results ---
                 topic                                        search_text  \
0   Rayleigh sky model  Rayleigh sky model: The Rayleigh sky model des...   
1            Water sky  Water sky: Water sky is a phenomenon that is c...   
2           Cyanometer  Cyanometer: A cyanometer (from cyan and -meter...   
3            Blue moon  Blue moon: A blue moon is an additional full m...   
4  Blue skies research  Blue skies research: Blue skies research (also...   

   _distance  
0   0.383766  
1   0.456188  
2   0.513083  
3   0.526569  
4   0.537272  


In [163]:
# Define search strategy functions with timing
from time import perf_counter
import pandas as pd

# Configuration
VECTOR_TOP_K = 100
FTS_TOP_K = 100
HYBRID_ALPHA = 0.5  # weight for text; (1 - alpha) for vector
FINAL_TOP_K = 3
RERANK_TOP_K = 50

def minmax_normalize(s):
    """Min-max normalization to [0,1]"""
    if s.max() == s.min():
        return (s * 0.0)
    return (s - s.min()) / (s.max() - s.min())

def vector_search(query, top_k=VECTOR_TOP_K):
    """
    Pure vector search using semantic embeddings.
    Args:
        query: str, search query
        top_k: int, number of results
    Returns:
        dict with 'results' (DataFrame), 'time_s' (float)
    """
    t0 = perf_counter()
    query_vector = model.encode(query).tolist()
    vec_df = table.search(query_vector).metric("cosine").limit(top_k).to_pandas()
    elapsed = perf_counter() - t0
    
    if '_distance' in vec_df.columns:
        vec_df['_vec_sim'] = 1.0 - vec_df['_distance']
    elif '_score' in vec_df.columns:
        vec_df['_vec_sim'] = vec_df['_score']
    else:
        vec_df['_vec_sim'] = 0.0
    
    show_cols = [c for c in ['id', 'topic', '_vec_sim'] if c in vec_df.columns]
    return {'results': vec_df[show_cols].head(FINAL_TOP_K), 'time_s': elapsed}

def fts_search(query, top_k=FTS_TOP_K):
    """
    Full-text search using keyword matching.
    Args:
        query: str, search query
        top_k: int, number of results
    Returns:
        dict with 'results' (DataFrame), 'time_s' (float)
    """
    t0 = perf_counter()
    fts_df = table.search(query, query_type="fts").limit(top_k).to_pandas()
    elapsed = perf_counter() - t0
    
    if '_score' in fts_df.columns:
        fts_df['_text_score'] = fts_df['_score']
    else:
        fts_df['_text_score'] = 0.0
    
    show_cols = [c for c in ['id', 'topic', '_text_score'] if c in fts_df.columns]
    return {'results': fts_df[show_cols].head(FINAL_TOP_K), 'time_s': elapsed}

def hybrid_search(query, alpha=HYBRID_ALPHA, top_k=FINAL_TOP_K):
    """
    Hybrid search combining vector and FTS with weighted scoring.
    Args:
        query: str, search query
        alpha: float, weight for text score (0-1); (1-alpha) for vector
        top_k: int, number of final results
    Returns:
        dict with 'results' (DataFrame), 'components' (dict of timings and raw dfs), 'time_s' (float)
    """
    t_start = perf_counter()
    
    # 1) Vector search
    t0 = perf_counter()
    query_vector = model.encode(query).tolist()
    vec_df = table.search(query_vector).metric("cosine").limit(VECTOR_TOP_K).to_pandas()
    t_vec = perf_counter() - t0
    
    if '_distance' in vec_df.columns:
        vec_df['_vec_sim_raw'] = 1.0 - vec_df['_distance']
    elif '_score' in vec_df.columns:
        vec_df['_vec_sim_raw'] = vec_df['_score']
    else:
        vec_df['_vec_sim_raw'] = 0.0
    
    # 2) FTS search
    t0 = perf_counter()
    fts_df = table.search(query, query_type="fts").limit(FTS_TOP_K).to_pandas()
    t_fts = perf_counter() - t0
    
    if '_score' in fts_df.columns:
        fts_df['_text_score_raw'] = fts_df['_score']
    else:
        fts_df['_text_score_raw'] = 0.0
    
    # 3) Merge and normalize
    t0 = perf_counter()
    merged = pd.merge(vec_df, fts_df, on='id', how='outer', suffixes=('_vec', '_fts'))
    merged['_vec_sim_raw'] = merged['_vec_sim_raw'].fillna(0.0)
    merged['_text_score_raw'] = merged['_text_score_raw'].fillna(0.0)
    
    merged['_vec_sim_norm'] = minmax_normalize(merged['_vec_sim_raw'])
    merged['_text_score_norm'] = minmax_normalize(merged['_text_score_raw'])
    merged['_hybrid_score'] = (1.0 - alpha) * merged['_vec_sim_norm'] + alpha * merged['_text_score_norm']
    t_merge = perf_counter() - t0
    
    # 4) Get top candidates
    results = merged.sort_values('_hybrid_score', ascending=False).head(top_k)
    show_cols = [c for c in ['id', 'topic_vec', '_vec_sim_raw', '_text_score_raw', '_hybrid_score'] if c in results.columns]
    
    total_time = perf_counter() - t_start
    return {
        'results': results[show_cols],
        'time_s': total_time,
        'components': {'vector_search_s': t_vec, 'fts_search_s': t_fts, 'merge_norm_s': t_merge}
    }

def hybrid_rerank_search(query, alpha=HYBRID_ALPHA, top_k=FINAL_TOP_K):
    """
    Hybrid search with CrossEncoder reranking for final ranking.
    Args:
        query: str, search query
        alpha: float, weight for text score (0-1)
        top_k: int, number of final results
    Returns:
        dict with 'results' (DataFrame), 'fallback' (bool), 'components' (timings), 'time_s' (float)
    """
    t_start = perf_counter()
    
    # First, run hybrid search to get candidates
    hybrid_result = hybrid_search(query, alpha=alpha, top_k=RERANK_TOP_K)
    t_hybrid = hybrid_result['time_s']
    rerank_pool = hybrid_result['results'].copy()
    
    # Attempt reranking
    fallback = False
    t_rerank = 0.0
    try:
        from sentence_transformers import CrossEncoder
        rerank_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
        
        # Prepare candidate texts
        if 'search_text_vec' in rerank_pool.columns:
            candidate_texts = rerank_pool['search_text_vec'].fillna('').tolist()
        elif 'search_text_fts' in rerank_pool.columns:
            candidate_texts = rerank_pool['search_text_fts'].fillna('').tolist()
        else:
            st_cols = [c for c in rerank_pool.columns if c.startswith('search_text')]
            candidate_texts = rerank_pool[st_cols[0]].fillna('').tolist() if st_cols else [''] * len(rerank_pool)
        
        # Rerank
        t0 = perf_counter()
        pairs = [(query, doc) for doc in candidate_texts]
        rerank_scores = rerank_model.predict(pairs)
        t_rerank = perf_counter() - t0
        
        rerank_pool['_rerank_score'] = rerank_scores
        results = rerank_pool.sort_values('_rerank_score', ascending=False).head(top_k)
        show_cols = [c for c in ['id', 'topic_vec', '_hybrid_score', '_rerank_score'] if c in results.columns]
    except Exception as e:
        print(f"⚠️ Reranker failed ({e}); falling back to hybrid results.")
        fallback = True
        results = rerank_pool.head(top_k)
        show_cols = [c for c in ['id', 'topic_vec', 'search_text_vec', '_hybrid_score'] if c in results.columns]
    
    total_time = perf_counter() - t_start
    return {
        'results': results[show_cols],
        'fallback': fallback,
        'time_s': total_time,
        'components': {**hybrid_result['components'], 'rerank_s': t_rerank}
    }

print("✅ Search functions defined: vector_search(), fts_search(), hybrid_search(), hybrid_rerank_search()")

✅ Search functions defined: vector_search(), fts_search(), hybrid_search(), hybrid_rerank_search()


In [175]:
# Demo: Call different search strategies with a single query
test_query = "Why is winter cold?"

print(f"Query: {test_query}\n")
print("=" * 80)

# 1) Vector search
print("\n1️⃣ VECTOR SEARCH")
vec_result = vector_search(test_query)
print(vec_result['results'])
print(f"⏱️ Time: {vec_result['time_s']:.4f}s")

# 2) FTS search
print("\n2️⃣ FULL-TEXT SEARCH (FTS)")
fts_result = fts_search(test_query)
print(fts_result['results'])
print(f"⏱️ Time: {fts_result['time_s']:.4f}s")

# 3) Hybrid search
print("\n3️⃣ HYBRID SEARCH (Vector + FTS combined)")
hybrid_result = hybrid_search(test_query)
print(hybrid_result['results'])
print(f"⏱️ Total Time: {hybrid_result['time_s']:.4f}s")
print(f"   Breakdown: vector={hybrid_result['components']['vector_search_s']:.4f}s, fts={hybrid_result['components']['fts_search_s']:.4f}s, merge={hybrid_result['components']['merge_norm_s']:.4f}s")

# 4) Hybrid + Rerank search
print("\n4️⃣ HYBRID + CROSSENCODER RERANK")
rerank_result = hybrid_rerank_search(test_query)
print(rerank_result['results'])
fallback_msg = " (fallback to hybrid)" if rerank_result['fallback'] else ""
print(f"⏱️ Total Time: {rerank_result['time_s']:.4f}s{fallback_msg}")
print(f"   Breakdown: vector={rerank_result['components']['vector_search_s']:.4f}s, fts={rerank_result['components']['fts_search_s']:.4f}s, merge={rerank_result['components']['merge_norm_s']:.4f}s, rerank={rerank_result['components']['rerank_s']:.4f}s")

# Comparison
print("\n" + "=" * 80)
print("SUMMARY: Strategy Performance Comparison")
print(f"Vector Search:       {vec_result['time_s']:.4f}s")
print(f"FTS Search:          {fts_result['time_s']:.4f}s")
print(f"Hybrid Search:       {hybrid_result['time_s']:.4f}s")
print(f"Hybrid + Rerank:     {rerank_result['time_s']:.4f}s")

Query: Why is winter cold?


1️⃣ VECTOR SEARCH
         id                    topic  _vec_sim
0  19151199              Winter rest  0.577812
1     50049        Subarctic climate  0.539252
2  10362761  The Coldest Winter Ever  0.489735
⏱️ Time: 0.4790s

2️⃣ FULL-TEXT SEARCH (FTS)
         id                        topic  _text_score
0   8021518  Hot summer cold winter zone    19.097233
1  17724421         Winterization of oil    17.988251
2  25276298                     Shishira    17.389906
⏱️ Time: 0.0179s

3️⃣ HYBRID SEARCH (Vector + FTS combined)
           id                    topic_vec  _vec_sim_raw  _text_score_raw  \
43   19151199                  Winter rest      0.577812        16.259386   
122     50049            Subarctic climate      0.539252        16.043913   
169   8021518  Hot summer cold winter zone      0.403116        19.097233   

     _hybrid_score  
43        0.925700  
122       0.886691  
169       0.848830  
⏱️ Total Time: 0.1112s
   Breakdown: vector=0.0634s

In [176]:
# Construct test queries based on sampling from the dataset
import lancedb
import random

db = lancedb.connect("./lancedb_data")
table = db.open_table("wiki_science_v2_fixed")

# 1. 获取总行数并随机采样索引
total_rows = table.count_rows()
random_indices = random.sample(range(total_rows), 10)

# 2. 提取样本
# LanceDB 支持通过原始位置或直接转换为 pandas 后采样
samples = table.to_pandas().iloc[random_indices]

for i, row in samples.iterrows():
    print(f"--- Sample {i} ---")
    print(f"REAL_ID: {row['id']}")
    print(f"TOPIC: {row['topic']}")
    print(f"CONTENT: {row['search_text']}")
    print("-" * 30)

--- Sample 47915 ---
REAL_ID: 1673339
TOPIC: Hyperfocus
CONTENT: Hyperfocus: Hyperfocus is an intense form of mental concentration or visualization that focuses consciousness on a subject, topic, or task. In some individuals, various subjects or topics may also include daydreams, concepts, fiction, the imagination, and other objects of the mind. Hyperfocus on a certain subject can cause side-tracking away from assigned or important tasks. Psychiatrically, it is considered to be a symptom of ADHD together with inattention, and it has been proposed as a symptom of other conditions, such as autism spectrum disorder (ASD). Hyperfocus may bear a relationship to the concept of flow.
------------------------------
--- Sample 4118 ---
REAL_ID: 41779645
TOPIC: Felicity Huntingford
CONTENT: Felicity Huntingford: Felicity Anne Huntingford FRSE (born 17 June 1948) is an aquatic ecologist known for her work in fish behaviour. Huntingford's research interests include the aggression in sticklebacks a

In [186]:
import pandas as pd
import time

# 1. 填入刚才构造的 10 个测试用例
eval_data = [
    # Q1: Hyperfocus
    {"query": "hyperfocus ADHD flow attention", "expected_id": "1673339"},
    
    # Q2: Fish behavior
    {"query": "fish aggression aquatic welfare", "expected_id": "41779645"},
    
    # Q3: On Suicide
    {"query": "student suicide psychoanalytic Vienna", "expected_id": "53949511"},
    
    # Q4: Nature Reserve
    {"query": "south africa prehistoric reserve", "expected_id": "30509260"},
    
    # Q5: Andrea Berlin
    {"query": "Boston archaeology professor chair", "expected_id": "53465218"},
    
    # Q6: Inflatable Space
    {"query": "inflatable space structure transport", "expected_id": "4268620"},
    
    # Q7: Manual Alphabet
    {"query": "hungarian fingerspelling sign language", "expected_id": "50689751"},
    
    # Q8: Herpolhode
    {"query": "herpolhode curve angular velocity", "expected_id": "12496940"},
    
    # Q9: Product Analysis
    {"query": "product analysis design deliverables", "expected_id": "53635793"},
    
    # Q10: Osaka Mercantile Exchange
    {"query": "Osaka futures exchange merger", "expected_id": "3057838"}
]

FINAL_TOP_K = 5  # 模拟最终交给 LLM 的上下文数量

def run_retrieval_benchmark(eval_data):
    comparison_data = []
    
    # 映射你的四个封装函数
    search_methods = {
        "Vector": vector_search,
        "FTS": fts_search,
        "Hybrid": hybrid_search,
        "Hybrid_Rerank": hybrid_rerank_search
    }

    for item in eval_data:
        q = item['query']
        target = str(item['expected_id'])
        row_metrics = {"Query": q[:20] + "..."}
        
        for name, func in search_methods.items():
            # 调用你的函数，获取标准化返回格式
            output = func(q)
            results_df = output['results']
            latency = output['time_s']
            
            # 1. 计算排名 (Rank)
            # 检查 target 是否出现在返回的 FINAL_TOP_K 结果中
            current_ids = results_df['id'].astype(str).tolist()
            if target in current_ids:
                rank = current_ids.index(target) + 1
            else:
                rank = "MISS"
            
            row_metrics[f"{name}_Rank"] = rank
            row_metrics[f"{name}_Time"] = f"{latency*1000:.1f}ms"
            
        comparison_data.append(row_metrics)
    
    return pd.DataFrame(comparison_data)

In [187]:
benchmark_df = run_retrieval_benchmark(eval_data)
print(benchmark_df.to_markdown(index=False))

| Query                   | Vector_Rank   | Vector_Time   |   FTS_Rank | FTS_Time   |   Hybrid_Rank | Hybrid_Time   | Hybrid_Rerank_Rank   | Hybrid_Rerank_Time   |
|:------------------------|:--------------|:--------------|-----------:|:-----------|--------------:|:--------------|:---------------------|:---------------------|
| hyperfocus ADHD flow... | 1             | 708.4ms       |          1 | 23.4ms     |             1 | 160.1ms       | 1                    | 2639.9ms             |
| fish aggression aqua... | MISS          | 911.7ms       |          1 | 13.7ms     |             2 | 113.9ms       | MISS                 | 1926.5ms             |
| student suicide psyc... | 1             | 163.3ms       |          1 | 13.0ms     |             1 | 103.7ms       | 1                    | 1819.2ms             |
| south africa prehist... | 2             | 134.7ms       |          1 | 19.6ms     |             1 | 113.2ms       | 1                    | 1778.4ms             |
| Boston archaeo